# 🎬 Maddələr Əsaslı Birgə Filtrləmə ilə Şəxsi Tövsiyə Sistemi
### (Item-Based Collaborative Filtering Recommendation Engine)

Əvvəlki notebookda işində biz tək bir film (*Star Wars*) üçün ona oxşar digər filmləri tapmağı öyrəndik. İndi isə bu yanaşmanı bir pillə irəli apararaq **istənilən istifadəçinin bütün izləmə tarixçəsinə və zövqünə əsaslanan tamfunksiyalı, fərdiləşdirilmiş Tövsiyə Mühərriki (Personalized Recommender System)** quracağıq!

---

### 💡 Maddələr Əsaslı Birgə Filtrləmənin (Item-Based CF) Əsas Məntiqi:
1. **Bütün Filmlərarası Oxşarlıq Matrisi (Item-Item Similarity Matrix):** Bütün filmlərin bir-biri ilə olan statistik korrelyasiyasını əvvəlcədən hesablayırıq.
2. **İstifadəçinin Zövq Profili:** İstifadəçinin əvvəllər izlədiyi və qiymətləndirdiyi filmləri çıxarırıq.
3. **Çəkiləndirilmiş Oxşarlıq Hesabı (Weighted Similarity Scoring):**
   - İstifadəçinin **5 ulduz verdiyi** sevimli filmlərə oxşar olan ekran əsərlərini böyük çəki ilə (x 5) mükafatlandırırıq.
   - İstifadəçinin **1 ulduz verdiyi** bəyənmədiyi filmlərə oxşar ekran əsərlərini isə kiçik çəki ilə (və ya mənfi təsirlə) cəzalandırırıq.
4. **Namizədlərin Birləşdirilməsi və Qruplaşdırılması (groupby().sum()):** Əgər bir film istifadəçinin bəyəndiyi bir neçə filmə eyni vaxtda bənzəyirsə, onun ümumi tövsiyə xalı toplanaraq ən yüksək pilləyə qalxır.
5. **Artıq İzlənilmiş Filmlərin Kənarlaşdırılması:** İstifadəçinin artıq baxıb qiymətləndirdiyi filmləri tövsiyə siyahısından çıxarırıq (çünki istifadəçiyə artıq bildiyi filmi yenidən tövsiyə etmək faydasızdır).

## 📂 1. Məlumat Dəstinin (MovieLens 100K) Yüklənməsi və Birləşdirilməsi

GroupLens tədqiqat qrupunun məşhur **MovieLens 100K** verilənlər bazasından istifadə edirik:
- ml-100k/u.data: 943 istifadəçinin 1682 filmə verdiyi 100,000 qiymətləndirmə (sütunlar: user_id, movie_id, 
ating).
- ml-100k/u.item: Filmlərin ID-si və adı (sütunlar: movie_id, 	itle).

Hər iki cədvəli movie_id sütununa əsasən birləşdirərək vahid 
atings DataFrame-i formalaşdırırıq.

In [ ]:
import pandas as pd

# u.data faylından istifadəçi ID, film ID və qiymətləndirmə sütunlarını oxuyuruq:
r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, usecols=range(3), encoding="ISO-8859-1")

# u.item faylından film ID və adını oxuyuruq:
m_cols = ['movie_id', 'title']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=m_cols, usecols=range(2), encoding="ISO-8859-1")

# 'movie_id' əsasında cədvəlləri birləşdiririk:
ratings = pd.merge(movies, ratings)

ratings.head()

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3


## 🗂️ 2. İstifadəçi-Film Qiymətləndirmə Matrisinin Qurulması (pivot_table)

Cədvəli **pivot_table** vasitəsilə 2-ölçülü seyrək matris (Sparse Matrix) formasına salırıq:
- **Sətirlər (Index):** user_id (istifadəçilər).
- **Sütunlar (Columns):** 	itle (bütün 1682 filmin adı).
- **Xanaların dəyəri (Values):** 
ating (istifadəçinin verdiyi 1-5 arası bal).

> 💡 **Qeyd:** Əksər xanalarda NaN (boşluq) olması təbiidir, çünki istifadəçilər yalnız izlədikləri filmlərə rəy bildiriblər. Tövsiyə sistemlərində bu hal **Seyrək Matris (Sparse Matrix)** adlanır.

In [2]:
# İstifadəçi-Film matrisini qururuq:
userRatings = ratings.pivot_table(index=['user_id'], columns=['title'], values='rating')
userRatings.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## ⚡ 3. Bütün Filmlərarası Korrelyasiya Matrisinin Hesablanması (corr())

Pandas-ın güclü daxili **.corr()** metodu matrisdəki bütün sütun cütləri (yəni hər bir film cütü) arasında **Pirson Korrelyasiyasını (Pearson Correlation)** hesablayır!

- Matrisdə 1682 film olduğu üçün, ümumilikdə  \times 1682 \approx 2.83\text{ milyon}$ cütlük üçün korrelyasiya hesablanır.
- Əgər iki filmi eyni vaxtda heç olmasa bir istifadəçi belə ortaq qiymətləndirməyibsə, nəticə NaN olur.

In [3]:
# Bütün film cütləri arasında korrelyasiya matrisini hesablayırıq:
corrMatrix = userRatings.corr()
corrMatrix

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
title,,,,,,,,,,,,,,,,,,,,,
'Til There Was You (1997),1.0,NaN,-1.000000e+00,-0.500000,-0.500000,0.522233,NaN,-4.264014e-01,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1-900 (1994),NaN,1.0,NaN,NaN,NaN,NaN,NaN,-9.819805e-01,NaN,NaN,...,NaN,NaN,NaN,-0.944911,NaN,NaN,NaN,NaN,NaN,NaN
101 Dalmatians (1996),-1.0,NaN,1.000000e+00,-0.049890,0.269191,0.048973,0.266928,-4.340657e-02,NaN,0.111111,...,NaN,-1.000000,NaN,0.158840,0.119234,0.680414,-4.875600e-17,0.707107,NaN,NaN
12 Angry Men (1957),-0.5,NaN,-4.989024e-02,1.000000,0.666667,0.256625,0.274772,1.788483e-01,NaN,0.457176,...,NaN,NaN,NaN,0.096546,0.068944,-0.361961,1.443376e-01,1.000000,1.0,NaN
187 (1997),-0.5,NaN,2.691910e-01,0.666667,1.000000,0.596644,NaN,-5.547002e-01,NaN,1.000000,...,NaN,0.866025,NaN,0.455233,-0.500000,0.500000,4.753271e-01,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Young Guns II (1990),NaN,NaN,6.804138e-01,-0.361961,0.500000,0.132017,-0.518476,-1.547646e-01,NaN,-0.054554,...,NaN,NaN,NaN,0.355001,0.722460,1.000000,8.660254e-01,NaN,NaN,NaN
"Young Poisoner's Handbook, The (1995)",NaN,NaN,-4.875600e-17,0.144338,0.475327,0.204926,0.623795,-3.417534e-01,NaN,0.707107,...,NaN,NaN,NaN,-0.413197,-0.019672,0.866025,1.000000e+00,NaN,NaN,NaN
Zeus and Roxanne (1997),NaN,NaN,7.071068e-01,1.000000,NaN,NaN,NaN,-1.000000e+00,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN


## 🛡️ 4. Yalançı Korrelyasiyaların Kənarlaşdırılması (min_periods=100)

Əvvəlki dərsdə gördüyümüz kimi, cəmi 1-2 istifadəçinin təsadüfən eyni qiymət verdiyi naməlum filmlər arasında riyazi olaraq 1.0 yalançı korrelyasiya (Spurious Correlation) yaranır.

Bunun qarşısını almaq üçün **corr()** metoduna **min_periods=100** parametrini ötürürük:
- **Mənası:** Yalnız **ən azı 100 ortaq istifadəçi tərəfindən qiymətləndirilmiş** film cütləri üçün korrelyasiya hesablansın!
- 100-dən az ortaq rəyi olan cütlüklər avtomatik olaraq NaN kimi qeyd edilərək kənarlaşdırılacaq.
- Bu parametr nəticələrin həm statistik etibarlılığını təmin edir, həm də tanınmış və populyar filmlərin tövsiyə edilməsinə şərait yaradır.

In [5]:
# Yalnız ən azı 100 ortaq qiymətləndirməsi olan film cütlüklərini saxlayırıq:
corrMatrix = userRatings.corr(method='pearson', min_periods=100)
corrMatrix

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
title,,,,,,,,,,,,,,,,,,,,,
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1-900 (1994),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
101 Dalmatians (1996),NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12 Angry Men (1957),NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187 (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Young Guns II (1990),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Young Poisoner's Handbook, The (1995)",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zeus and Roxanne (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 👤 5. Sınaq İstifadəçisi (User ID: 0) üçün Zövq Profilinin Çıxarılması

Verilənlər bazasına test məqsədilə xüsusi bir sınaq istifadəçisi — **user_id = 0** əlavə edilmişdir.

Gəlin bu istifadəçinin zövqünə nəzər salaq:
- userRatings.loc[0] sətrini götürürük və dropna() vasitəsilə izləmədiyi filmləri kənarlaşdırırıq.
- İstifadəçinin rəyləri:
  - 🌟 **The Empire Strikes Back (1980)**: 5.0 (çox bəyənib)
  - 🌟 **Star Wars (1977)**: 5.0 (çox bəyənib)
  - 💔 **Gone with the Wind (1939)**: 1.0 (qətiyyən bəyənməyib / nifrət edib)

Gördüyünüz kimi, bu istifadəçi elmi-fantastik kosmik dastanları çox sevir, klassik tarixi dram və melodram janrlarından isə xoşu gəlmir.

In [9]:
# User ID 0-ın qiymətləndirdiyi filmləri Series kimi çıxarırıq:
myRatings = userRatings.loc[0].dropna()
myRatings

title
Empire Strikes Back, The (1980)    5.0
Gone with the Wind (1939)          1.0
Star Wars (1977)                   5.0
Name: 0, dtype: float64

## 🎯 6. Fərdiləşdirilmiş Tövsiyə Namizədlərinin Çəkiləndirilməsi (Weighting & Scaling)

İndi istifadəçinin qiymətləndirdiyi hər bir film üzrə tək-tək dövr edirik:
1. corrMatrix-dən həmin filmə oxşar olan bütün digər filmləri və onların korrelyasiya xallarını götürürük (sims).
2. **Xalların Çəkiləndirilməsi (Scaling):**
   - Həmin oxşarlıq xallarını **istifadəçinin bu filmə verdiyi bal ilə vururuq** (sims.map(lambda x: x * myRatings[i])).
   - Məsələn: İstifadəçinin 5 bal verdiyi *Star Wars* və *Empire Strikes Back* filmlərinə bənzəyən ekran əsərləri $\times 5$ bal qazanır!
   - İstifadəçinin cəmi 1 bal verdiyi *Gone with the Wind*-ə bənzəyən filmlər isə cəmi $\times 1$ çəki alır (mənfi korrelyasiya olduqda isə ümumi balı azaldır).
3. Hər bir filmdən yaranan namizədləri vahid simCandidates siyahısına əlavə edirik (pd.concat).

In [24]:
simCandidates = pd.Series(dtype='float64')

for i in range(0, len(myRatings.index)):
    movie_title = myRatings.index[i]
    rating_val = myRatings.iloc[i]
    print(f"Adding sims for {movie_title} (Rating: {rating_val})...")
    
    # Bu filmə oxşar filmlərin siyahısını korrelyasiya matrisindən götürürük:
    sims = corrMatrix[movie_title].dropna()
    
    # Oxşarlıq əmsalını istifadəçinin verdiyi reytinqlə vuraraq çəkiləndiririk:
    sims = sims.map(lambda x: x * rating_val)
    
    # Əldə olunan namizədləri ümumi namizədlər siyahısına əlavə edirik:
    simCandidates = pd.concat([simCandidates, sims])

print("Sorting...")
simCandidates.sort_values(inplace=True, ascending=False)
simCandidates.head(20)

Adding sims for Empire Strikes Back, The (1980)...
Adding sims for Gone with the Wind (1939)...
Adding sims for Star Wars (1977)...
sorting...


Empire Strikes Back, The (1980)                       5.000000
Star Wars (1977)                                      5.000000
Empire Strikes Back, The (1980)                       3.741763
Star Wars (1977)                                      3.741763
Return of the Jedi (1983)                             3.606146
Return of the Jedi (1983)                             3.362779
Raiders of the Lost Ark (1981)                        2.693297
Raiders of the Lost Ark (1981)                        2.680586
Austin Powers: International Man of Mystery (1997)    1.887164
Sting, The (1973)                                     1.837692
Bridge on the River Kwai, The (1957)                  1.783717
Indiana Jones and the Last Crusade (1989)             1.750535
Cinderella (1950)                                     1.749598
Back to the Future (1985)                             1.726427
Terminator 2: Judgment Day (1991)                     1.667662
Frighteners, The (1996)                               1

## ➕ 7. Eyni Namizədlərin Xallarının Toplanması (groupby().sum())

Yuxarıdakı ilkin nəticələrə diqqət yetirsəniz, bəzi filmlərin siyahıda bir neçə dəfə təkrarlandığını görərsiniz:
- Məsələn: *Return of the Jedi (1983)* həm *Star Wars*, həm də *The Empire Strikes Back* filminə bənzədiyi üçün hər ikisindən yüksək çəki qazanıb.

Bu təbiidir və tövsiyə sistemləri üçün böyük üstünlükdür! Əgər bir film istifadəçinin bəyəndiyi **bir neçə sevimli filmin hər ikisinə eyni vaxtda bənzəyirsə**, onun ümumi tövsiyə dəyəri daha yüksək olmalıdır.

Buna görə də **groupby().sum()** tətbiq edərək təkrarlanan filmlərin xallarını cəmləyirik.

In [36]:
# Təkrarlanan filmlərin oxşarlıq xallarını cəmləyirik:
simCandidates = simCandidates.groupby(simCandidates.index).sum()
simCandidates.sort_values(inplace=True, ascending=False)
simCandidates

Empire Strikes Back, The (1980)              8.877450
Star Wars (1977)                             8.870971
Return of the Jedi (1983)                    7.178172
Raiders of the Lost Ark (1981)               5.519700
Indiana Jones and the Last Crusade (1989)    3.488028
                                               ...   
Annie Hall (1977)                           -0.511775
Real Genius (1985)                          -0.552871
Remains of the Day, The (1993)              -0.560337
This Is Spinal Tap (1984)                   -0.636474
First Wives Club, The (1996)                -0.972480
Length: 268, dtype: float64

## 🚫 8. Artıq İzlənilmiş Filmlərin Kənarlaşdırılması (Filtering Out Watched Movies)

Cədvəlin ən başında ən yüksək xallarla istifadəçinin artıq izlədiyi filmlər durur (*The Empire Strikes Back* və *Star Wars*).

Lakin istifadəçiyə artıq baxdığı və 5 ulduz verdiyi filmi yenidən "Bunu izləyin" deyə tövsiyə etmək heç bir fayda vermir. Buna görə də **.drop(myRatings.index)** ilə istifadəçinin artıq qiymətləndirdiyi filmləri tövsiyə siyahısından çıxarırıq.

In [37]:
# İstifadəçinin artıq qiymətləndirdiyi filmləri siyahıdan kənarlaşdırırıq:
filteredSims = simCandidates.drop(myRatings.index)
filteredSims

Return of the Jedi (1983)                    7.178172
Raiders of the Lost Ark (1981)               5.519700
Indiana Jones and the Last Crusade (1989)    3.488028
Bridge on the River Kwai, The (1957)         3.366616
Back to the Future (1985)                    3.357941
                                               ...   
Annie Hall (1977)                           -0.511775
Real Genius (1985)                          -0.552871
Remains of the Day, The (1993)              -0.560337
This Is Spinal Tap (1984)                   -0.636474
First Wives Club, The (1996)                -0.972480
Length: 265, dtype: float64

In [ ]:
# 🏆 İstifadəçiyə təqdim ediləcək Ən Yaxşı 10 Fərdiləşdirilmiş Tövsiyə (Top-10 Recommendations):
top10 = filteredSims.head(10)

print("=" * 75)
print("🎬 User ID 0 üçün Ən Yaxşı 10 Fərdiləşdirilmiş Film Tövsiyəsi:")
print("=" * 75)
for rank, (title, score) in enumerate(top10.items(), 1):
    print(f"{rank:2d}. {title:<52} [Tövsiyə Xalı: {score:.2f}]")
print("=" * 75)

# --- Nəticələrin Təhlili: ---
# 1. 'Return of the Jedi (1983)' [Xal: 7.18] - Star Wars klassik trilogiyasının 3-cü filmi!
# 2. 'Raiders of the Lost Ark (1981)' [Xal: 5.52] - Corc Lukas və Harrison Ford tandemi (Indiana Jones)!
# 3. 'Indiana Jones and the Last Crusade (1989)' [Xal: 3.49] - Macəra-fantastika klassikası!
# 4. 'Bridge on the River Kwai, The (1957)' və 'Back to the Future (1985)' [Xal: ~3.36]!
# Cədvəlin ən aşağısında isə mənfi xallarla istifadəçinin bəyənmədiyi 'Gone with the Wind' kimi dram/romantika filmləri yer alır.